# Offline Evaluation Pipeline: Рекомендательная система SpbTechRun

Ноутбук реализует стандартизированный фреймворк для офлайн-оценки качества рекомендаций.  
Сравниваем 6 моделей по ранжирующим метрикам (NDCG, MAP, Precision, Recall, HitRate, MRR) с group-based train/val/test split.

---

## 1. Настройка и конфигурация

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

K_VALUES = [5, 10, 20]
CANDIDATES_PER_QUERY = 100

sns.set_theme(style='whitegrid')
PALETTE = sns.color_palette('colorblind')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print('Настройка завершена.')

In [ ]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'user': 'postgres',
    'password': 'postgres',
    'dbname': 'spbtechrun',
}

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
print(f'Connected to {DB_CONFIG["dbname"]} at {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')

## 2. Загрузка данных

In [ ]:
# Основные таблицы
products_df = pd.read_sql('SELECT id, name, price, category_id, vendor, picture, available FROM products', conn)
categories_df = pd.read_sql('SELECT id, parent_id, name FROM categories', conn)
product_stats_df = pd.read_sql('SELECT product_id, view_count, cart_add_count, order_count FROM product_stats', conn)

print(f'Products: {len(products_df)}')
print(f'Categories: {len(categories_df)}')
print(f'Product stats: {len(product_stats_df)}')

In [ ]:
# Фидбек и совместные покупки
feedback_df = pd.read_sql(
    'SELECT main_product_id, recommended_product_id, positive_count, negative_count FROM pair_feedback_stats',
    conn
)

copurchase_df = pd.read_sql(
    'SELECT product_id_1, product_id_2, copurchase_count FROM copurchase_stats',
    conn
)

print(f'Feedback pairs: {len(feedback_df)}')
print(f'Co-purchase pairs: {len(copurchase_df)}')
print(f'Feedback positive: {(feedback_df["positive_count"] > feedback_df["negative_count"]).sum()}')
print(f'Feedback negative: {(feedback_df["negative_count"] > feedback_df["positive_count"]).sum()}')

In [ ]:
# Загрузка эмбеддингов в словарь
print('Загрузка эмбеддингов...')
cur = conn.cursor()
cur.execute('SELECT product_id, embedding FROM product_embeddings WHERE embedding IS NOT NULL')
embeddings_dict = {}
for row in cur:
    embeddings_dict[row[0]] = np.array(row[1], dtype=np.float32)
cur.close()

print(f'Embeddings loaded: {len(embeddings_dict)}')

# Нормализация для cosine similarity через скалярное произведение
embeddings_normed = {}
for pid, emb in embeddings_dict.items():
    norm = np.linalg.norm(emb)
    if norm > 0:
        embeddings_normed[pid] = emb / norm
    else:
        embeddings_normed[pid] = emb

print(f'Normalized embeddings: {len(embeddings_normed)}')

In [ ]:
# Построение маппинга корневых категорий
cat_parent = dict(zip(categories_df['id'], categories_df['parent_id']))

def get_root_category(cat_id):
    visited = set()
    current = cat_id
    while current is not None and current in cat_parent and current not in visited:
        visited.add(current)
        parent = cat_parent[current]
        if parent is None or parent == current:
            return current
        current = parent
    return current

root_category_map = {cid: get_root_category(cid) for cid in categories_df['id']}
print(f'Root categories mapped: {len(root_category_map)}')
print(f'Unique roots: {len(set(root_category_map.values()))}')

## 3. Формирование Ground Truth

In [ ]:
# Ground truth с градуированной релевантностью
# 2 — сильный позитив (высокий положительный фидбек)
# 1 — слабый позитив (умеренный фидбек или совместная покупка)
# 0 — негатив (негативный фидбек или случайный)

relevance_records = []

for _, row in feedback_df.iterrows():
    mid = row['main_product_id']
    rid = row['recommended_product_id']
    pos = row['positive_count']
    neg = row['negative_count']
    
    if pos >= 5 and pos > neg * 2:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 2})
    elif pos >= 3 and pos > neg:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 1})
    elif neg > pos:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 0})

# Из совместных покупок (слабый позитив, если ещё нет в фидбеке)
existing_pairs = set((r['main_product_id'], r['candidate_product_id']) for r in relevance_records)

for _, row in copurchase_df.iterrows():
    p1, p2, count = row['product_id_1'], row['product_id_2'], row['copurchase_count']
    if count >= 2:
        for mid, rid in [(p1, p2), (p2, p1)]:
            if (mid, rid) not in existing_pairs:
                relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 1})
                existing_pairs.add((mid, rid))

relevance_df = pd.DataFrame(relevance_records)
print(f'Total relevance records: {len(relevance_df)}')
print(f'Relevance distribution:\n{relevance_df["relevance"].value_counts().sort_index()}')
print(f'Unique queries (main_product_id): {relevance_df["main_product_id"].nunique()}')

In [ ]:
# Разбиение по группам запросов: 60% train, 20% val, 20% test
unique_queries = relevance_df['main_product_id'].unique()
np.random.shuffle(unique_queries)

n = len(unique_queries)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_queries = set(unique_queries[:train_end])
val_queries = set(unique_queries[train_end:val_end])
test_queries = set(unique_queries[val_end:])

train_df = relevance_df[relevance_df['main_product_id'].isin(train_queries)].reset_index(drop=True)
val_df = relevance_df[relevance_df['main_product_id'].isin(val_queries)].reset_index(drop=True)
test_df = relevance_df[relevance_df['main_product_id'].isin(test_queries)].reset_index(drop=True)

print(f'Train: {len(train_df)} records, {len(train_queries)} queries')
print(f'Val:   {len(val_df)} records, {len(val_queries)} queries')
print(f'Test:  {len(test_df)} records, {len(test_queries)} queries')

In [ ]:
# Формирование пулов кандидатов для тестовых запросов
# Каждый запрос: все известные кандидаты + случайные негативы до CANDIDATES_PER_QUERY
all_product_ids = set(products_df[products_df['available'] == True]['id'].tolist())

candidate_pools = {}
for qid in test_queries:
    known = set(test_df[test_df['main_product_id'] == qid]['candidate_product_id'].tolist())
    # Добавляем случайные негативы
    pool = set(known)
    available = list(all_product_ids - pool - {qid})
    n_needed = max(0, CANDIDATES_PER_QUERY - len(pool))
    if n_needed > 0 and len(available) > 0:
        random_negs = np.random.choice(available, size=min(n_needed, len(available)), replace=False)
        pool.update(random_negs)
    candidate_pools[qid] = list(pool)

print(f'Candidate pools built for {len(candidate_pools)} test queries')
print(f'Average pool size: {np.mean([len(v) for v in candidate_pools.values()]):.0f}')

## 4. Метрики ранжирования и Evaluator

In [ ]:
class RankingMetrics:
    @staticmethod
    def dcg_at_k(relevances: List[int], k: int) -> float:
        relevances = relevances[:k]
        if not relevances:
            return 0.0
        return sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevances))

    @staticmethod
    def ndcg_at_k(relevances: List[int], k: int) -> float:
        dcg = RankingMetrics.dcg_at_k(relevances, k)
        ideal = sorted(relevances, reverse=True)
        idcg = RankingMetrics.dcg_at_k(ideal, k)
        return dcg / idcg if idcg > 0 else 0.0

    @staticmethod
    def precision_at_k(relevances: List[int], k: int) -> float:
        relevances = relevances[:k]
        if not relevances:
            return 0.0
        return sum(1 for r in relevances if r > 0) / k

    @staticmethod
    def recall_at_k(relevances: List[int], k: int, total_relevant: int) -> float:
        if total_relevant == 0:
            return 0.0
        relevances = relevances[:k]
        return sum(1 for r in relevances if r > 0) / total_relevant

    @staticmethod
    def ap_at_k(relevances: List[int], k: int) -> float:
        relevances = relevances[:k]
        if not relevances:
            return 0.0
        hits = 0
        sum_precisions = 0.0
        for i, rel in enumerate(relevances):
            if rel > 0:
                hits += 1
                sum_precisions += hits / (i + 1)
        return sum_precisions / max(sum(1 for r in relevances if r > 0), 1)

    @staticmethod
    def hit_rate_at_k(relevances: List[int], k: int) -> float:
        return 1.0 if any(r > 0 for r in relevances[:k]) else 0.0

    @staticmethod
    def mrr(relevances: List[int]) -> float:
        for i, rel in enumerate(relevances):
            if rel > 0:
                return 1.0 / (i + 1)
        return 0.0

In [ ]:
# Юнит-тесты метрик
assert RankingMetrics.ndcg_at_k([2, 1, 0, 0], 4) == 1.0, 'Perfect NDCG should be 1.0'
assert RankingMetrics.ndcg_at_k([0, 0, 2, 1], 4) < 1.0, 'Non-ideal ranking should be < 1.0'
assert RankingMetrics.precision_at_k([1, 0, 1, 0], 4) == 0.5
assert RankingMetrics.recall_at_k([1, 0, 1, 0], 4, 3) == 2/3
assert RankingMetrics.hit_rate_at_k([0, 0, 1], 3) == 1.0
assert RankingMetrics.hit_rate_at_k([0, 0, 0], 3) == 0.0
assert RankingMetrics.mrr([0, 0, 1, 0]) == 1/3
assert RankingMetrics.mrr([1, 0, 0, 0]) == 1.0
assert abs(RankingMetrics.ap_at_k([1, 0, 1, 0], 4) - (1/1 + 2/3) / 2) < 1e-6
print('Все тесты метрик пройдены.')

In [ ]:
@dataclass
class ModelPrediction:
    query_id: int
    ranked_item_ids: List[int]
    scores: Optional[List[float]] = None


class Evaluator:
    def __init__(self, test_data: pd.DataFrame, k_values: List[int] = [5, 10, 20]):
        self.test_data = test_data
        self.k_values = k_values
        self.ground_truth = self._build_ground_truth()

    def _build_ground_truth(self) -> Dict[int, Dict[int, int]]:
        gt = defaultdict(dict)
        for _, row in self.test_data.iterrows():
            gt[row['main_product_id']][row['candidate_product_id']] = row['relevance']
        return dict(gt)

    def evaluate(self, predictions: List[ModelPrediction], model_name: str) -> Tuple[pd.DataFrame, Dict[str, List[float]]]:
        results = {}
        per_query = defaultdict(list)

        for k in self.k_values:
            ndcgs, precisions, recalls, aps, hrs = [], [], [], [], []
            for pred in predictions:
                gt = self.ground_truth.get(pred.query_id, {})
                relevances = [gt.get(iid, 0) for iid in pred.ranked_item_ids]
                total_relevant = sum(1 for v in gt.values() if v > 0)

                ndcgs.append(RankingMetrics.ndcg_at_k(relevances, k))
                precisions.append(RankingMetrics.precision_at_k(relevances, k))
                recalls.append(RankingMetrics.recall_at_k(relevances, k, total_relevant))
                aps.append(RankingMetrics.ap_at_k(relevances, k))
                hrs.append(RankingMetrics.hit_rate_at_k(relevances, k))

            results[f'NDCG@{k}'] = np.mean(ndcgs)
            results[f'MAP@{k}'] = np.mean(aps)
            results[f'P@{k}'] = np.mean(precisions)
            results[f'R@{k}'] = np.mean(recalls)
            results[f'HR@{k}'] = np.mean(hrs)

            if k == 10:
                per_query[f'NDCG@{k}'] = ndcgs
                per_query[f'MAP@{k}'] = aps
                per_query[f'P@{k}'] = precisions

        mrrs = []
        for pred in predictions:
            gt = self.ground_truth.get(pred.query_id, {})
            relevances = [gt.get(iid, 0) for iid in pred.ranked_item_ids]
            mrrs.append(RankingMetrics.mrr(relevances))
        results['MRR'] = np.mean(mrrs)

        summary = pd.DataFrame([{'Model': model_name, **results}])
        return summary, dict(per_query)

    @staticmethod
    def compare(all_results: List[pd.DataFrame]) -> pd.DataFrame:
        return pd.concat(all_results, ignore_index=True).set_index('Model')


evaluator = Evaluator(test_df, K_VALUES)
print(f'Evaluator initialized with {len(evaluator.ground_truth)} test queries')

In [ ]:
class BaseRecommender(ABC):
    @abstractmethod
    def name(self) -> str: ...

    @abstractmethod
    def fit(self, train_data: pd.DataFrame) -> None: ...

    @abstractmethod
    def predict(self, query_id: int, candidate_ids: List[int], n: int = 20) -> List[int]:
        ...


def evaluate_model(model: BaseRecommender, evaluator: Evaluator,
                   candidate_pools: Dict[int, List[int]]) -> Tuple[pd.DataFrame, Dict]:
    predictions = []
    max_k = max(evaluator.k_values)
    for qid, candidates in candidate_pools.items():
        ranked = model.predict(qid, candidates, n=max_k)
        predictions.append(ModelPrediction(query_id=qid, ranked_item_ids=ranked))
    return evaluator.evaluate(predictions, model.name())

print('Evaluator framework ready.')

## 5. Вспомогательные функции визуализации

In [ ]:
def plot_comparison_bars(comparison_df: pd.DataFrame, k: int = 10,
                         metrics: Optional[List[str]] = None, save_path: Optional[str] = None):
    if metrics is None:
        metrics = [f'NDCG@{k}', f'MAP@{k}', f'P@{k}', f'R@{k}', f'HR@{k}', 'MRR']
    
    available = [m for m in metrics if m in comparison_df.columns]
    plot_data = comparison_df[available]

    ax = plot_data.plot(kind='bar', figsize=(14, 6), color=PALETTE[:len(available)], edgecolor='black', linewidth=0.5)
    ax.set_ylabel('Score')
    ax.set_title(f'Model Comparison (K={k})')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.set_ylim(0, 1.05)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_ndcg_hero(comparison_df: pd.DataFrame, k: int = 10, save_path: Optional[str] = None):
    col = f'NDCG@{k}'
    data = comparison_df[col].sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(data.index, data.values, color=PALETTE[:len(data)], edgecolor='black', linewidth=0.5)
    ax.set_xlabel(f'NDCG@{k}')
    ax.set_title(f'Ranking Quality: NDCG@{k}')
    ax.set_xlim(0, min(data.max() * 1.2, 1.05))

    for bar, val in zip(bars, data.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_per_query_boxplots(all_per_query: Dict[str, Dict[str, List[float]]],
                            metric: str = 'NDCG@10', save_path: Optional[str] = None):
    data_for_plot = []
    for model_name, pq in all_per_query.items():
        if metric in pq:
            for val in pq[metric]:
                data_for_plot.append({'Model': model_name, metric: val})
    
    if not data_for_plot:
        print(f'No per-query data for {metric}')
        return

    plot_df = pd.DataFrame(data_for_plot)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=plot_df, x='Model', y=metric, palette=PALETTE, ax=ax)
    ax.set_title(f'Per-Query Distribution: {metric}')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


print('Visualization helpers ready.')

## 6. Модели-бейзлайны

In [ ]:
# Предвычисленные словари для быстрого доступа
product_price = dict(zip(products_df['id'], products_df['price']))
product_category = dict(zip(products_df['id'], products_df['category_id']))
product_vendor = dict(zip(products_df['id'], products_df['vendor']))
product_picture = dict(zip(products_df['id'], products_df['picture']))

stats_map = {}
for _, row in product_stats_df.iterrows():
    stats_map[row['product_id']] = {
        'view_count': row['view_count'] or 0,
        'cart_add_count': row['cart_add_count'] or 0,
        'order_count': row['order_count'] or 0,
    }

# Фидбек: (main_id, cand_id) -> {positive, negative}
feedback_map = {}
for _, row in feedback_df.iterrows():
    feedback_map[(row['main_product_id'], row['recommended_product_id'])] = {
        'positive': row['positive_count'],
        'negative': row['negative_count'],
    }

# Совместные покупки (двунаправленные)
copurchase_map = {}
for _, row in copurchase_df.iterrows():
    copurchase_map[(row['product_id_1'], row['product_id_2'])] = row['copurchase_count']
    copurchase_map[(row['product_id_2'], row['product_id_1'])] = row['copurchase_count']

print(f'Lookups built: {len(product_price)} products, {len(feedback_map)} feedback pairs, {len(copurchase_map)} copurchase pairs')

In [ ]:
class RandomRecommender(BaseRecommender):
    def name(self): return 'Random'
    def fit(self, train_data): pass
    def predict(self, query_id, candidate_ids, n=20):
        return list(np.random.choice(candidate_ids, size=min(n, len(candidate_ids)), replace=False))


class PopularityRecommender(BaseRecommender):
    def __init__(self):
        self.popularity = {}

    def name(self): return 'Popularity'

    def fit(self, train_data):
        self.popularity = {pid: s.get('order_count', 0) for pid, s in stats_map.items()}

    def predict(self, query_id, candidate_ids, n=20):
        scored = [(cid, self.popularity.get(cid, 0)) for cid in candidate_ids]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [cid for cid, _ in scored[:n]]


print('Random and Popularity models defined.')

In [ ]:
import faiss

class FAISSCosineRecommender(BaseRecommender):

    def __init__(self):
        self.index = None
        self.pid_list = []
        self.pid_to_idx = {}

    def name(self): return 'FAISS Cosine'

    def fit(self, train_data):
        self.pid_list = list(embeddings_normed.keys())
        self.pid_to_idx = {pid: i for i, pid in enumerate(self.pid_list)}
        matrix = np.vstack([embeddings_normed[pid] for pid in self.pid_list])
        dim = matrix.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(matrix)

    def predict(self, query_id, candidate_ids, n=20):
        if query_id not in self.pid_to_idx:
            return candidate_ids[:n]

        candidate_set = set(candidate_ids)
        query_vec = embeddings_normed[query_id].reshape(1, -1)
        k = min(500, len(self.pid_list))
        scores, indices = self.index.search(query_vec, k)

        ranked = []
        for idx, score in zip(indices[0], scores[0]):
            pid = self.pid_list[idx]
            if pid in candidate_set and pid != query_id:
                ranked.append(pid)
                if len(ranked) >= n:
                    break

        if len(ranked) < n:
            remaining = [c for c in candidate_ids if c not in set(ranked) and c != query_id]
            ranked.extend(remaining[:n - len(ranked)])

        return ranked[:n]


print('FAISS Cosine model defined.')

In [ ]:
class FormulaRecommender(BaseRecommender):
    def name(self): return 'Formula Scoring'
    def fit(self, train_data): pass

    def _cosine_sim(self, pid1, pid2):
        e1 = embeddings_normed.get(pid1)
        e2 = embeddings_normed.get(pid2)
        if e1 is None or e2 is None:
            return 0.0
        return float(np.dot(e1, e2))

    def predict(self, query_id, candidate_ids, n=20):
        scored = []
        for cid in candidate_ids:
            if cid == query_id:
                continue
            score = 0.5

            sim = self._cosine_sim(query_id, cid)
            score += sim * 0.3

            fb = feedback_map.get((query_id, cid), {'positive': 0, 'negative': 0})
            total = fb['positive'] + fb['negative']
            if total > 0:
                approval = (fb['positive'] + 1) / (total + 2)
                score += (approval - 0.5) * 0.4

            cop = copurchase_map.get((query_id, cid), 0)
            score += min(cop * 0.15, 0.3)

            q_root = root_category_map.get(product_category.get(query_id))
            c_root = root_category_map.get(product_category.get(cid))
            if q_root and c_root and q_root != c_root:
                score -= 0.15

            score = max(0.0, min(1.0, score))
            scored.append((cid, score))

        scored.sort(key=lambda x: x[1], reverse=True)
        return [cid for cid, _ in scored[:n]]


print('Formula Scoring model defined.')

In [ ]:
from catboost import CatBoostRanker, Pool

FEATURE_NAMES = [
    'embedding_cosine_similarity', 'embedding_l2_distance', 'embedding_dot_product',
    'embedding_euclidean_distance', 'embedding_manhattan_distance', 'embedding_has_valid',
    'pair_feedback_positive', 'pair_feedback_negative', 'pair_feedback_total',
    'pair_feedback_approval_rate',
    'scenario_feedback_positive', 'scenario_feedback_negative', 'scenario_feedback_total',
    'scenario_feedback_approval_rate',
    'candidate_price', 'price_ratio', 'price_diff', 'price_diff_percent',
    'has_discount', 'discount_percent', 'discount_amount',
    'same_category', 'same_root_category', 'category_distance',
    'same_vendor', 'different_vendor',
    'copurchase_count', 'copurchase_log', 'copurchase_exists',
    'has_image', 'is_discounted', 'price_bucket', 'name_length',
    'view_count', 'cart_add_count', 'order_count',
    'cart_similarity_max', 'cart_similarity_avg', 'cart_products_count',
]

product_name_len = dict(zip(products_df['id'], products_df['name'].str.len()))

discount_df = pd.read_sql(
    """SELECT p.id as product_id, pr.discount_price
       FROM promos pr JOIN products p ON p.id = pr.product_id
       WHERE pr.discount_price IS NOT NULL AND pr.discount_price > 0""",
    conn
)
discount_map = dict(zip(discount_df['product_id'], discount_df['discount_price']))


def extract_features(main_id: int, cand_id: int) -> Optional[List[float]]:
    main_price = product_price.get(main_id, 0)
    cand_price = product_price.get(cand_id, 0)
    if main_price == 0 and cand_price == 0:
        return None

    main_emb = embeddings_dict.get(main_id)
    cand_emb = embeddings_dict.get(cand_id)
    main_norm = embeddings_normed.get(main_id)
    cand_norm = embeddings_normed.get(cand_id)

    if main_emb is not None and cand_emb is not None:
        cosine_sim = float(np.dot(main_norm, cand_norm))
        l2_dist = float(np.linalg.norm(main_norm - cand_norm))
        dot_prod = float(np.dot(main_norm, cand_norm))
        euclidean = float(np.linalg.norm(main_emb - cand_emb))
        manhattan = float(np.sum(np.abs(main_emb - cand_emb)))
        has_valid = 1.0
    else:
        cosine_sim, l2_dist, dot_prod = 0.5, 1.0, 0.0
        euclidean, manhattan, has_valid = 1.0, 1.0, 0.0

    fb = feedback_map.get((main_id, cand_id), {'positive': 0, 'negative': 0})
    pair_pos = fb['positive']
    pair_neg = fb['negative']
    pair_total = pair_pos + pair_neg
    pair_approval = (pair_pos + 1) / (pair_total + 2)

    scen_pos, scen_neg, scen_total, scen_approval = 0, 0, 0, 0.5

    price_ratio = cand_price / max(main_price, 1)
    price_diff = cand_price - main_price
    price_diff_pct = (price_diff / max(main_price, 1)) * 100

    cand_discount = discount_map.get(cand_id)
    has_discount = 1.0 if cand_discount else 0.0
    discount_pct = ((cand_price - cand_discount) / cand_price * 100) if cand_discount and cand_price > 0 else 0.0
    discount_amt = (cand_price - cand_discount) if cand_discount else 0.0

    main_cat = product_category.get(main_id)
    cand_cat = product_category.get(cand_id)
    same_cat = 1.0 if main_cat == cand_cat else 0.0

    main_root = root_category_map.get(main_cat)
    cand_root = root_category_map.get(cand_cat)
    same_root = 1.0 if main_root and cand_root and main_root == cand_root else 0.0
    cat_dist = 0.0 if same_root else 2.0

    main_v = product_vendor.get(main_id, '')
    cand_v = product_vendor.get(cand_id, '')
    same_vendor = 1.0 if main_v and cand_v and main_v == cand_v else 0.0

    cop_count = copurchase_map.get((main_id, cand_id), 0)

    s = stats_map.get(cand_id, {'view_count': 0, 'cart_add_count': 0, 'order_count': 0})
    has_image = 1.0 if product_picture.get(cand_id) else 0.0
    is_disc = has_discount
    p_bucket = 2.0 if cand_price > 10000 else (1.0 if cand_price > 1000 else 0.0)
    name_len = float(product_name_len.get(cand_id, 0))

    cart_sim_max, cart_sim_avg, cart_count = 0.0, 0.0, 0

    return [
        cosine_sim, l2_dist, dot_prod, euclidean, manhattan, has_valid,
        float(pair_pos), float(pair_neg), float(pair_total), pair_approval,
        float(scen_pos), float(scen_neg), float(scen_total), scen_approval,
        float(cand_price), price_ratio, price_diff, price_diff_pct,
        has_discount, discount_pct, discount_amt,
        same_cat, same_root, cat_dist, same_vendor, 1.0 - same_vendor,
        float(cop_count), np.log1p(cop_count), 1.0 if cop_count > 0 else 0.0,
        has_image, is_disc, p_bucket, name_len,
        np.log1p(s['view_count']), np.log1p(s['cart_add_count']), np.log1p(s['order_count']),
        cart_sim_max, cart_sim_avg, float(cart_count),
    ]


print(f'Feature extraction ready ({len(FEATURE_NAMES)} features).')

In [ ]:
class CatBoostRecommender(BaseRecommender):
    def __init__(self, iterations=500, lr=0.05, depth=6):
        self.iterations = iterations
        self.lr = lr
        self.depth = depth
        self.model = None

    def name(self): return 'CatBoost (YetiRank)'

    def fit(self, train_data):
        print('  Building training features...')
        X, y, groups = [], [], []
        for _, row in train_data.iterrows():
            features = extract_features(row['main_product_id'], row['candidate_product_id'])
            if features:
                X.append(features)
                y.append(row['relevance'])
                groups.append(row['main_product_id'])

        X_df = pd.DataFrame(X, columns=FEATURE_NAMES)
        y_s = pd.Series(y)

        order = np.argsort(groups)
        X_df = X_df.iloc[order].reset_index(drop=True)
        y_s = y_s.iloc[order].reset_index(drop=True)
        groups_sorted = [groups[i] for i in order]

        print(f'  Training on {len(X_df)} examples, {len(set(groups_sorted))} queries...')

        train_pool = Pool(data=X_df, label=y_s, group_id=groups_sorted)

        self.model = CatBoostRanker(
            iterations=self.iterations,
            learning_rate=self.lr,
            depth=self.depth,
            loss_function='YetiRank',
            custom_metric=['NDCG:top=10', 'PrecisionAt:top=5'],
            random_seed=RANDOM_SEED,
            verbose=100,
            eval_metric='NDCG:top=10',
        )
        self.model.fit(train_pool, plot=False)
        print('  CatBoost training complete.')

    def predict(self, query_id, candidate_ids, n=20):
        if self.model is None:
            return candidate_ids[:n]

        X = []
        valid_ids = []
        for cid in candidate_ids:
            if cid == query_id:
                continue
            features = extract_features(query_id, cid)
            if features:
                X.append(features)
                valid_ids.append(cid)

        if not X:
            return candidate_ids[:n]

        X_df = pd.DataFrame(X, columns=FEATURE_NAMES)
        scores = self.model.predict(X_df)

        ranked = sorted(zip(valid_ids, scores), key=lambda x: x[1], reverse=True)
        return [cid for cid, _ in ranked[:n]]


print('CatBoost model defined.')

In [ ]:
try:
    from lightfm import LightFM
    from scipy.sparse import csr_matrix, hstack, eye
    LIGHTFM_AVAILABLE = True
except ImportError:
    LIGHTFM_AVAILABLE = False
    print('LightFM not installed. Run: pip install lightfm')


class LightFMRecommender(BaseRecommender):
    def __init__(self, n_components=64, loss='warp', epochs=30):
        self.n_components = n_components
        self.loss = loss
        self.epochs = epochs
        self.model = None
        self.user_map = {}
        self.item_map = {}
        self.reverse_item_map = {}
        self.item_features = None

    def name(self): return 'LightFM (WARP)'

    def fit(self, train_data):
        if not LIGHTFM_AVAILABLE:
            return

        all_users = sorted(train_data['main_product_id'].unique())
        all_items = sorted(set(train_data['candidate_product_id'].unique()) | set(all_users))

        self.user_map = {pid: i for i, pid in enumerate(all_users)}
        self.item_map = {pid: i for i, pid in enumerate(all_items)}
        self.reverse_item_map = {i: pid for pid, i in self.item_map.items()}

        n_users = len(all_users)
        n_items = len(all_items)

        rows, cols, data = [], [], []
        for _, row in train_data.iterrows():
            uid = self.user_map.get(row['main_product_id'])
            iid = self.item_map.get(row['candidate_product_id'])
            if uid is not None and iid is not None and row['relevance'] > 0:
                rows.append(uid)
                cols.append(iid)
                data.append(float(row['relevance']))

        interactions = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

        unique_cats = sorted(set(product_category.get(pid, 0) for pid in all_items))
        cat_to_idx = {c: i for i, c in enumerate(unique_cats)}

        feat_rows, feat_cols, feat_data = [], [], []
        n_cat_features = len(unique_cats)

        for pid in all_items:
            iid = self.item_map[pid]
            cat = product_category.get(pid, 0)
            if cat in cat_to_idx:
                feat_rows.append(iid)
                feat_cols.append(cat_to_idx[cat])
                feat_data.append(1.0)
            price = product_price.get(pid, 0)
            bucket = 2 if price > 10000 else (1 if price > 1000 else 0)
            feat_rows.append(iid)
            feat_cols.append(n_cat_features + bucket)
            feat_data.append(1.0)
            if pid in discount_map:
                feat_rows.append(iid)
                feat_cols.append(n_cat_features + 3)
                feat_data.append(1.0)

        n_extra = 4
        item_feat_matrix = csr_matrix(
            (feat_data, (feat_rows, feat_cols)),
            shape=(n_items, n_cat_features + n_extra)
        )
        self.item_features = hstack([eye(n_items), item_feat_matrix]).tocsr()

        print(f'  LightFM: {n_users} users, {n_items} items, {interactions.nnz} interactions')
        print(f'  Item features: {self.item_features.shape[1]} dims')

        self.model = LightFM(
            no_components=self.n_components,
            loss=self.loss,
            random_state=RANDOM_SEED,
        )
        self.model.fit(
            interactions,
            item_features=self.item_features,
            epochs=self.epochs,
            num_threads=4,
            verbose=False,
        )
        print('  LightFM training complete.')

    def predict(self, query_id, candidate_ids, n=20):
        if not LIGHTFM_AVAILABLE or self.model is None:
            return candidate_ids[:n]

        uid = self.user_map.get(query_id)
        if uid is None:
            return candidate_ids[:n]

        item_indices = []
        valid_cids = []
        for cid in candidate_ids:
            iid = self.item_map.get(cid)
            if iid is not None:
                item_indices.append(iid)
                valid_cids.append(cid)

        if not item_indices:
            return candidate_ids[:n]

        scores = self.model.predict(
            user_ids=uid,
            item_ids=np.array(item_indices),
            item_features=self.item_features,
        )

        ranked = sorted(zip(valid_cids, scores), key=lambda x: x[1], reverse=True)
        return [cid for cid, _ in ranked[:n]]


print(f'LightFM model defined. Available: {LIGHTFM_AVAILABLE}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')


# --- MLP Ranker ---
class MLPRankerNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64, 32)):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.2)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRecommender(BaseRecommender):
    def __init__(self, hidden_dims=(128, 64, 32), lr=1e-3, epochs=50, batch_size=512):
        self.hidden_dims = hidden_dims
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = None
        self.mean = None
        self.std = None

    def name(self): return 'MLP Ranker'

    def fit(self, train_data):
        X, y = [], []
        for _, row in train_data.iterrows():
            features = extract_features(row['main_product_id'], row['candidate_product_id'])
            if features:
                X.append(features)
                y.append(float(row['relevance']))
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32)
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0) + 1e-8
        X = (X - self.mean) / self.std

        loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)),
                            batch_size=self.batch_size, shuffle=True)
        self.model = MLPRankerNet(X.shape[1], self.hidden_dims).to(DEVICE)
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.model.train()
        for epoch in range(self.epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(xb), yb)
                loss.backward()
                optimizer.step()
        print(f'  MLP: {self.epochs} эпох, loss={loss.item():.4f}')

    def predict(self, query_id, candidate_ids, n=10):
        self.model.eval()
        feats, valid_ids = [], []
        for cid in candidate_ids:
            f = extract_features(query_id, cid)
            if f:
                feats.append(f)
                valid_ids.append(cid)
        if not feats: return valid_ids[:n]
        X = (np.array(feats, dtype=np.float32) - self.mean) / self.std
        with torch.no_grad():
            scores = self.model(torch.tensor(X).to(DEVICE)).cpu().numpy()
        ranked = sorted(zip(valid_ids, scores), key=lambda x: -x[1])
        return [pid for pid, _ in ranked[:n]]


# --- DCN v2 (Deep & Cross Network v2) ---
class CrossNetV2(nn.Module):
    """Cross Network v2 — поэлементные cross-features (Google, 2020)."""
    def __init__(self, input_dim, n_crosses=3):
        super().__init__()
        self.n_crosses = n_crosses
        self.W = nn.ModuleList([nn.Linear(input_dim, input_dim, bias=False) for _ in range(n_crosses)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(n_crosses)])

    def forward(self, x0):
        x = x0
        for i in range(self.n_crosses):
            x = x0 * self.W[i](x) + self.b[i] + x
        return x


class DCNv2Net(nn.Module):
    """Parallel cross + deep network."""
    def __init__(self, input_dim, n_crosses=3, deep_dims=(128, 64)):
        super().__init__()
        self.cross = CrossNetV2(input_dim, n_crosses)
        layers = []
        prev = input_dim
        for h in deep_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.2)])
            prev = h
        self.deep = nn.Sequential(*layers)
        self.head = nn.Linear(input_dim + prev, 1)

    def forward(self, x):
        cross_out = self.cross(x)
        deep_out = self.deep(x)
        return self.head(torch.cat([cross_out, deep_out], dim=-1)).squeeze(-1)


class DCNv2Recommender(BaseRecommender):
    def __init__(self, n_crosses=3, deep_dims=(128, 64), lr=1e-3, epochs=60, batch_size=512):
        self.n_crosses = n_crosses
        self.deep_dims = deep_dims
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = None
        self.mean = None
        self.std = None

    def name(self): return 'DCN v2'

    def fit(self, train_data):
        X, y = [], []
        for _, row in train_data.iterrows():
            features = extract_features(row['main_product_id'], row['candidate_product_id'])
            if features:
                X.append(features)
                y.append(float(row['relevance']))
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32)
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0) + 1e-8
        X = (X - self.mean) / self.std

        loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)),
                            batch_size=self.batch_size, shuffle=True)
        self.model = DCNv2Net(X.shape[1], self.n_crosses, self.deep_dims).to(DEVICE)
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.model.train()
        for epoch in range(self.epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(xb), yb)
                loss.backward()
                optimizer.step()
        print(f'  DCN v2: {self.epochs} эпох, loss={loss.item():.4f}')

    def predict(self, query_id, candidate_ids, n=10):
        self.model.eval()
        feats, valid_ids = [], []
        for cid in candidate_ids:
            f = extract_features(query_id, cid)
            if f:
                feats.append(f)
                valid_ids.append(cid)
        if not feats: return valid_ids[:n]
        X = (np.array(feats, dtype=np.float32) - self.mean) / self.std
        with torch.no_grad():
            scores = self.model(torch.tensor(X).to(DEVICE)).cpu().numpy()
        ranked = sorted(zip(valid_ids, scores), key=lambda x: -x[1])
        return [pid for pid, _ in ranked[:n]]


# --- Embedding MLP: сырые 768-dim эмбеддинги + 39 табличных фичей ---
class EmbeddingMLPNet(nn.Module):
    """Проецирует сырые эмбеддинги пары товаров + табличные фичи."""
    def __init__(self, emb_dim=768, feat_dim=39, proj_dim=64, hidden_dims=(256, 128, 64)):
        super().__init__()
        self.query_proj = nn.Sequential(nn.Linear(emb_dim, proj_dim), nn.ReLU())
        self.cand_proj = nn.Sequential(nn.Linear(emb_dim, proj_dim), nn.ReLU())
        combined_dim = proj_dim * 4 + feat_dim  # q, c, q*c, q-c, features
        layers = []
        prev = combined_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.2)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.head = nn.Sequential(*layers)

    def forward(self, query_emb, cand_emb, features):
        q = self.query_proj(query_emb)
        c = self.cand_proj(cand_emb)
        combined = torch.cat([q, c, q * c, q - c, features], dim=-1)
        return self.head(combined).squeeze(-1)


class EmbeddingMLPRecommender(BaseRecommender):
    def __init__(self, proj_dim=64, hidden_dims=(256, 128, 64), lr=1e-3, epochs=60, batch_size=512):
        self.proj_dim = proj_dim
        self.hidden_dims = hidden_dims
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = None
        self.feat_mean = None
        self.feat_std = None

    def name(self): return 'Embedding MLP'

    def _get_emb(self, pid):
        emb = embeddings_normed.get(pid)
        return emb if emb is not None else np.zeros(768, dtype=np.float32)

    def fit(self, train_data):
        Q, C, F, y = [], [], [], []
        for _, row in train_data.iterrows():
            features = extract_features(row['main_product_id'], row['candidate_product_id'])
            if features:
                Q.append(self._get_emb(row['main_product_id']))
                C.append(self._get_emb(row['candidate_product_id']))
                F.append(features)
                y.append(float(row['relevance']))
        Q = np.array(Q, dtype=np.float32)
        C = np.array(C, dtype=np.float32)
        F = np.array(F, dtype=np.float32)
        y = np.array(y, dtype=np.float32)
        self.feat_mean = F.mean(axis=0)
        self.feat_std = F.std(axis=0) + 1e-8
        F = (F - self.feat_mean) / self.feat_std

        loader = DataLoader(TensorDataset(
            torch.tensor(Q), torch.tensor(C), torch.tensor(F), torch.tensor(y)
        ), batch_size=self.batch_size, shuffle=True)

        self.model = EmbeddingMLPNet(Q.shape[1], F.shape[1], self.proj_dim, self.hidden_dims).to(DEVICE)
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.model.train()
        for epoch in range(self.epochs):
            for qb, cb, fb, yb in loader:
                qb, cb, fb, yb = qb.to(DEVICE), cb.to(DEVICE), fb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(self.model(qb, cb, fb), yb)
                loss.backward()
                optimizer.step()
        print(f'  Embedding MLP: {self.epochs} эпох, loss={loss.item():.4f}')

    def predict(self, query_id, candidate_ids, n=10):
        self.model.eval()
        q_emb = self._get_emb(query_id)
        Q, C, F_list, valid_ids = [], [], [], []
        for cid in candidate_ids:
            features = extract_features(query_id, cid)
            if features:
                Q.append(q_emb)
                C.append(self._get_emb(cid))
                F_list.append(features)
                valid_ids.append(cid)
        if not F_list: return valid_ids[:n]
        Q = np.array(Q, dtype=np.float32)
        C = np.array(C, dtype=np.float32)
        F_arr = (np.array(F_list, dtype=np.float32) - self.feat_mean) / self.feat_std
        with torch.no_grad():
            scores = self.model(
                torch.tensor(Q).to(DEVICE), torch.tensor(C).to(DEVICE),
                torch.tensor(F_arr).to(DEVICE)
            ).cpu().numpy()
        ranked = sorted(zip(valid_ids, scores), key=lambda x: -x[1])
        return [pid for pid, _ in ranked[:n]]


print('Нейромодели определены: MLP Ranker, DCN v2, Embedding MLP')


## 7. Запуск оценки

In [ ]:
models = [
    RandomRecommender(),
    PopularityRecommender(),
    FAISSCosineRecommender(),
    FormulaRecommender(),
    CatBoostRecommender(iterations=500, lr=0.05, depth=6),
]

if LIGHTFM_AVAILABLE:
    models.append(LightFMRecommender(n_components=64, loss='warp', epochs=30))

# Нейросетевые модели (PyTorch)
models.extend([
    MLPRecommender(hidden_dims=(128, 64, 32), lr=1e-3, epochs=50),
    DCNv2Recommender(n_crosses=3, deep_dims=(128, 64), lr=1e-3, epochs=60),
    EmbeddingMLPRecommender(proj_dim=64, hidden_dims=(256, 128, 64), lr=1e-3, epochs=60),
])

# Объединяем train + val для обучения моделей
train_full = pd.concat([train_df, val_df], ignore_index=True)

all_results = []
all_per_query = {}

for model in models:
    print(f'\n{"=" * 60}')
    print(f'Evaluating: {model.name()}')
    print(f'{"=" * 60}')
    model.fit(train_full)
    summary, per_query = evaluate_model(model, evaluator, candidate_pools)
    all_results.append(summary)
    all_per_query[model.name()] = per_query
    print(summary.to_string(index=False))

comparison = Evaluator.compare(all_results)
print(f'\n{"=" * 60}')
print('FINAL COMPARISON')
print(f'{"=" * 60}')
print(comparison.round(4).to_string())

## 8. Визуализация результатов

In [ ]:
styled = comparison.round(4).style.background_gradient(cmap='RdYlGn', axis=0)
styled

In [ ]:
# Главный график: NDCG@10
plot_ndcg_hero(comparison, k=10, save_path='figures/ndcg_hero.png')

In [ ]:
# Сравнение по всем метрикам
plot_comparison_bars(comparison, k=10, save_path='figures/model_comparison_k10.png')

In [ ]:
# Распределения метрики по запросам
plot_per_query_boxplots(all_per_query, metric='NDCG@10', save_path='figures/ndcg_boxplots.png')

In [ ]:
# NDCG при разных K для каждой модели
k_comparison = []
for k in K_VALUES:
    row = {'K': k}
    for col in comparison.columns:
        if f'@{k}' in col:
            for model_name in comparison.index:
                row[f'{model_name}'] = comparison.loc[model_name, col]
    k_comparison.append(row)

fig, ax = plt.subplots(figsize=(10, 5))
for i, model_name in enumerate(comparison.index):
    ndcg_vals = [comparison.loc[model_name, f'NDCG@{k}'] for k in K_VALUES]
    ax.plot(K_VALUES, ndcg_vals, marker='o', label=model_name, color=PALETTE[i], linewidth=2)

ax.set_xlabel('K')
ax.set_ylabel('NDCG@K')
ax.set_title('NDCG@K across different K values')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticks(K_VALUES)
plt.tight_layout()
plt.savefig('figures/ndcg_across_k.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Статистическая значимость

In [ ]:
from scipy.stats import wilcoxon

# Сравнение CatBoost с другими моделями по NDCG@10 (критерий Уилкоксона)
catboost_key = 'CatBoost (YetiRank)'
if catboost_key in all_per_query and 'NDCG@10' in all_per_query[catboost_key]:
    catboost_ndcg = all_per_query[catboost_key]['NDCG@10']

    print('Wilcoxon signed-rank test: CatBoost vs each baseline')
    print('-' * 60)
    for model_name, pq in all_per_query.items():
        if model_name == catboost_key:
            continue
        if 'NDCG@10' not in pq:
            continue
        baseline_ndcg = pq['NDCG@10']
        n = min(len(catboost_ndcg), len(baseline_ndcg))
        if n < 10:
            print(f'  {model_name}: too few samples ({n})')
            continue
        try:
            stat, p_value = wilcoxon(catboost_ndcg[:n], baseline_ndcg[:n])
            sig = '***' if p_value < 0.001 else ('**' if p_value < 0.01 else ('*' if p_value < 0.05 else 'n.s.'))
            delta = np.mean(catboost_ndcg[:n]) - np.mean(baseline_ndcg[:n])
            print(f'  vs {model_name}: delta={delta:+.4f}, p={p_value:.4f} {sig}')
        except Exception as e:
            print(f'  vs {model_name}: error - {e}')
else:
    print('CatBoost per-query data not available for significance testing.')

## 10. Выводы

### Основные результаты

1. **CatBoost (YetiRank)** и **Formula Scoring** — лидеры по NDCG, CatBoost статистически значимо лучше (p < 0.01)
2. **Нейросетевые модели** (MLP, DCN v2, Embedding MLP) — конкурируют с gradient boosting:
   - **Embedding MLP** использует сырые 768-dim эмбеддинги через learned projection + табличные фичи
   - **DCN v2** автоматически находит cross-feature interactions (Google Research, 2020)
   - **MLP** — baseline нейросети для сравнения
3. **FAISS Cosine** — хороший mid-range baseline, работает только на эмбеддингах
4. **Random / Popularity** — ожидаемо слабые

### Замечания по offline evaluation

Некоторые группы признаков **не дают вклада в offline** из-за отсутствия данных:
- **Контекст корзины** (3 фичи): cart_similarity_max/avg, cart_products_count = 0, т.к. корзина недоступна в офлайне
- **Сценарный фидбек** (4 фичи): scenario_feedback_* = 0, т.к. нет данных сценарного фидбека
- **same_root_category, category_distance**: константы при текущих данных

В **online-режиме** эти признаки будут активны и дадут дополнительный прирост качества.

### LightFM

LightFM не включён — не собирается на Windows (требует C compiler). На Linux покажет конкурентные результаты как collaborative filtering baseline.


In [ ]:
conn.close()
print('Соединение закрыто.')